In [84]:
!pip install playwright pandas beautifulsoup4 requests tqdm nest_asyncio
!playwright install

In [85]:
import requests
import pandas as pd
import nest_asyncio

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

from tqdm import tqdm
from playwright.async_api import async_playwright

In [86]:
nest_asyncio.apply()

In [87]:
from urllib.parse import urljoin, urlparse

async def crawl_site_playwright(start_url, max_pages=100):

    visited = set()
    queue = [start_url]
    pages = []

    domain = urlparse(start_url).netloc

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        while queue and len(pages) < max_pages:

            url = queue.pop(0)

            if url in visited:
                continue

            visited.add(url)
            pages.append(url)

            print("Crawling:", url)

            try:

                await page.goto(url, timeout=60000)

                # grab all links from rendered page
                links = await page.eval_on_selector_all(
                    "a[href]",
                    "elements => elements.map(e => e.href)"
                )

                for link in links:

                    parsed = urlparse(link)

                    if parsed.netloc == domain:

                        clean_url = parsed.scheme + "://" + parsed.netloc + parsed.path

                        if clean_url not in visited:
                            queue.append(clean_url)

            except Exception as e:

                print("Error:", e)

        await browser.close()

    return list(set(pages))

In [88]:
async def scan_page(page, url):

    await page.goto(url)

    await page.add_script_tag(
        url="https://cdnjs.cloudflare.com/ajax/libs/axe-core/4.8.2/axe.min.js"
    )

    results = await page.evaluate(
        """async () => {
            return await axe.run();
        }"""
    )

    return results["violations"]

In [89]:
def parse_violations(page_url, violations):

    rows = []

    for v in violations:

        for node in v["nodes"]:

            rows.append({
                "page": page_url,
                "violation": v["id"],
                "severity": v["impact"],
                "description": v["description"],
                "content": node["html"],
                "recommended_fix": v["help"]
            })

    return rows

In [90]:
pages = await crawl_site_playwright("https://www.honolulupd.org", max_pages=100)

print("Pages discovered:", len(pages))
pages

Crawling: https://www.honolulupd.org
Crawling: https://www.honolulupd.org/
Crawling: https://www.honolulupd.org/organization/
Crawling: https://www.honolulupd.org/information/
Crawling: https://www.honolulupd.org/public-affairs-office/
Crawling: https://www.honolulupd.org/community-programs/
Crawling: https://www.honolulupd.org/police-services/
Crawling: https://www.honolulupd.org/about-us/
Crawling: https://www.honolulupd.org/safer-roads-together/
Crawling: https://www.honolulupd.org/information/pedestrian-safety/
Crawling: https://www.honolulupd.org/if-you-wait-until-you-celebrate-its-too-late-plan-ahead-for-a-side-ride/
Crawling: https://www.honolulupd.org/pal/
Crawling: https://www.honolulupd.org/hpd-shuts-down-illegal-gambling-operation-on-mala-street-in-wahiawa-2/
Crawling: https://www.honolulupd.org/help-us-keep-emergency-lines-open/
Crawling: https://www.honolulupd.org/hpd-promotion-ceremony-march-11-2026/
Crawling: https://www.honolulupd.org/news/
Crawling: https://www.honolul

['https://www.honolulupd.org/cad/',
 'https://www.honolulupd.org/didyouknow/',
 'https://www.honolulupd.org/community-programs/drug-abuse-resistance-training-d-a-r-e/',
 'https://www.honolulupd.org/community-programs/law-enforcement-explorers-program-l-e-e-p/',
 'https://www.honolulupd.org/community-programs/ride-along-program/',
 'https://www.honolulupd.org/pathways/faq/',
 'https://www.honolulupd.org/help-us-keep-emergency-lines-open/',
 'https://www.honolulupd.org/wp-content/uploads/2024/02/Pathways-Internship-Program-Brochure-4.pdf',
 'https://www.honolulupd.org/organization/patrol-districts/district-4/',
 'https://www.honolulupd.org/hpd-shuts-down-illegal-gambling-operation-on-mala-street-in-wahiawa-2/',
 'https://www.honolulupd.org/cyber-crimes/',
 'https://www.honolulupd.org/organization/chief-of-police/',
 'https://www.honolulupd.org/information/covered-offender-registry/',
 'https://www.honolulupd.org/d6/',
 'https://www.honolulupd.org/pal/',
 'https://www.honolulupd.org/polic

In [91]:
async def run_scan(pages):

    all_rows = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for url in tqdm(pages):

            try:

                violations = await scan_page(page, url)

                rows = parse_violations(url, violations)

                all_rows.extend(rows)

            except Exception as e:

                print("Error scanning:", url)

        await browser.close()

    return all_rows

In [92]:
all_rows = await run_scan(pages)

  7%|█████▋                                                                            | 7/100 [00:17<02:54,  1.88s/it]

Error scanning: https://www.honolulupd.org/wp-content/uploads/2024/02/Pathways-Internship-Program-Brochure-4.pdf


 69%|███████████████████████████████████████████████████████▉                         | 69/100 [01:35<00:21,  1.42it/s]

Error scanning: https://www.honolulupd.org/wp-content/uploads/2026/01/Honolulu-Police-Department-2025-Legislative-Disciplinary-Report.pdf


100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [02:12<00:00,  1.32s/it]


In [93]:
df = pd.DataFrame(all_rows)

df.head()

,page,violation,severity,description,content,recommended_fix
0,https://www.honolulupd.org/cad/,aria-allowed-role,minor,Ensures role attribute has an appropriate valu...,"<li class=""kb-slide-item kb-gallery-slide-item...",ARIA role should be appropriate for the element
1,https://www.honolulupd.org/cad/,list,serious,Ensures that lists are structured correctly,"<ul id=""menu-sidebar-organization"" class=""menu...","<ul> and <ol> must only directly contain <li>,..."
2,https://www.honolulupd.org/community-programs/...,aria-allowed-role,minor,Ensures role attribute has an appropriate valu...,"<li class=""kb-slide-item kb-gallery-slide-item...",ARIA role should be appropriate for the element
3,https://www.honolulupd.org/community-programs/...,landmark-no-duplicate-banner,moderate,Ensures the document has at most one banner la...,"<header class=""site-header"" style=""margin-top:...",Document should not have more than one banner ...
4,https://www.honolulupd.org/community-programs/...,aria-allowed-role,minor,Ensures role attribute has an appropriate valu...,"<li class=""kb-slide-item kb-gallery-carousel-i...",ARIA role should be appropriate for the element


In [ ]:
df.to_csv("outputs/reports/wcag_report.csv", index=False)

In [ ]:
#errors

#outputs

#crawle
#TEXT LEFT JUSTIFIED
#NUMBERS RIGHT